In [1]:
import json
import os
# 允许存在多个 OpenMP 副本，强行解除限制
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
from typing import Dict, List
import pandas as pd
import numpy as np
import time
import openai
import re
import hashlib
import torch
import backtrader as bt
import quantstats as qs
import gc
import tushare as ts
import matplotlib
matplotlib.use('Agg') 


In [2]:

class InformationBase:
    def __init__(self, filepath: str = "information_base.json", max_memory_size: int = 15):
        self.filepath = filepath
        self.max_memory_size = max_memory_size
        self.memory = self._load_or_init()

    def _load_or_init(self) -> Dict[str, List[str]]:
        """加载或初始化信息库文件"""
        if os.path.exists(self.filepath):
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        else:
            initial_data = {
                "recommended_directions": ["尝试量价结合计算波动率。"],
                "prohibited_directions": ["禁止将绝对价格直接作为因子。"]
            }
            self._save(initial_data)
            return initial_data

    def _save(self, data: Dict = None):
        """持久化保存到 JSON"""
        data_to_save = data if data else self.memory
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, ensure_ascii=False, indent=4)

    def update_experience(self, action: str, content: str):
        """
        更新记忆，并自动维护队列长度 (FIFO)
        action: 'add_recommended', 'add_prohibited', 'add_failed_attempt'
        """
        key_map = {
            "add_recommended": "recommended_directions",
            "add_prohibited": "prohibited_directions"
        }
        target_key = key_map.get(action)
        if target_key:
            self.memory[target_key].append(content)
            # 截断早期记忆，防止 LLM Context 爆炸
            self.memory[target_key] = self.memory[target_key][-self.max_memory_size:]
            self._save()

    def get_prompt_summary(self) -> str:
        """生成供 LLM 读取的字符串摘要"""
        return json.dumps(self.memory, ensure_ascii=False, indent=2)

In [3]:
class FactorBase:
    def __init__(self, metadata_path: str = "factor_base.json", data_dir: str = "./factor_data/"):
        self.metadata_path = metadata_path
        self.data_dir = data_dir
        os.makedirs(self.data_dir, exist_ok=True)
        self.factors_metadata = self._load_or_init()

    def _load_or_init(self) -> List[Dict]:
        if os.path.exists(self.metadata_path):
            with open(self.metadata_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return[]

    def _save_metadata(self):
        with open(self.metadata_path, 'w', encoding='utf-8') as f:
            json.dump(self.factors_metadata, f, ensure_ascii=False, indent=4)

    def add_factor(self, formula: str, metrics: Dict, factor_df: pd.DataFrame):
        """入库新因子：保存元数据并落盘数据"""
        factor_id = f"alpha_{len(self.factors_metadata) + 1:04d}"
        
        # 1. 保存元数据
        new_record = {
            "factor_id": factor_id,
            "formula": formula,
            "metrics": metrics
        }
        self.factors_metadata.append(new_record)
        self._save_metadata()
        
        # 2. 保存计算好的 DataFrame 到本地 (使用 parquet 极速读写)
        file_path = os.path.join(self.data_dir, f"{factor_id}.parquet")
        factor_df.to_parquet(file_path)

    def get_all_factor_data(self) -> Dict[str, pd.DataFrame]:
        """加载所有已有因子的数据，用于正交化/相关性检验"""
        loaded_data = {}
        for factor in self.factors_metadata:
            fid = factor["factor_id"]
            file_path = os.path.join(self.data_dir, f"{fid}.parquet")
            if os.path.exists(file_path):
                loaded_data[fid] = pd.read_parquet(file_path)
        return loaded_data

    def get_prompt_summary(self) -> str:
        """只暴露公式给 LLM 避免过长"""
        formulas = [f["formula"] for f in self.factors_metadata]
        return "\n".join(formulas) if formulas else "暂无因子"

In [16]:
# 自动挂载 GPU (如果没有 GPU，PyTorch 也会使用 C++ 底层的 CPU 张量运算，依然比 Pandas 快上百倍)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 因子评测引擎已挂载至计算节点: {device}")

class EvaluationEngine:
    def __init__(self, stock_data: dict, forward_returns: pd.DataFrame, factor_base, benchmark_returns: pd.Series, pool_mask: pd.DataFrame = None):
        self.factor_base = factor_base
        self.index_dates = forward_returns.index
        self.columns_stocks = forward_returns.columns
        
        # 1. 内存全量驻留：将所有输入数据一次性转化为 GPU Tensor 矩阵
        self.stock_tensor = {k: torch.tensor(v.values, dtype=torch.float64, device=device) 
                             for k, v in stock_data.items()}
                             
        self.fwd_tensor = torch.tensor(forward_returns.values, dtype=torch.float64, device=device)
        self.bench_tensor = torch.tensor(benchmark_returns.reindex(self.index_dates).fillna(0).values, 
                                         dtype=torch.float64, device=device)
        if pool_mask is not None:
            # 保证日期和股票对齐
            aligned_mask = pool_mask.reindex(index=self.index_dates, columns=self.columns_stocks, fill_value=False)
            self.pool_mask_tensor = torch.tensor(aligned_mask.values, dtype=torch.bool, device=device)
        else:
            self.pool_mask_tensor = None

        # 为衰减测算预先准备的位移目标矩阵
        self.fwd_tensor_shift_1 = torch.cat([self.fwd_tensor[1:], torch.full((1, self.fwd_tensor.shape[1]), float('nan'), device=device)])
        self.fwd_tensor_shift_2 = torch.cat([self.fwd_tensor[2:], torch.full((2, self.fwd_tensor.shape[1]), float('nan'), device=device)])
        
        # 2. 注册 GPU 原生算子
        self.safe_env = self._register_operators()
        self.safe_env.update(self.stock_tensor)

    # ==========================================
    # 🧠 底层 GPU 张量数学算子核 (完全替换 Pandas)
    # ==========================================
    def _cs_rank(self, x):
        """GPU 极速横截面排序 (0~1)"""
        valid = ~torch.isnan(x)
        # 将 NaN 替换为极小值排到最前，最后再 Mask 掉
        x_filled = torch.where(valid, x, torch.tensor(-float('inf'), device=device))
        ranks = x_filled.argsort(dim=1).argsort(dim=1).to(torch.float32)
        count = valid.sum(dim=1, keepdim=True).to(torch.float32)
        ranks = ranks / (count - 1).clamp(min=1)
        return torch.where(valid, ranks, torch.tensor(float('nan'), device=device))

    def _ts_rank(self, x, d):
        """GPU 极速时序排序 (利用 unfold 进行时间窗口切片)"""
        res = torch.full_like(x, float('nan'))
        if x.shape[0] < d: return res
        unf = x.unfold(0, d, 1)  # [T-d+1, N, d]
        last_vals = unf[:, :, -1:]
        valid = ~torch.isnan(unf)
        count = valid.sum(dim=2).float()
        # 计算窗口内比最后一个元素小的数量
        ranks = (unf <= last_vals).logical_and(valid).sum(dim=2).float()
        pct_rank = ranks / count.clamp(min=1)
        res[d-1:] = torch.where(count > 0, pct_rank, float('nan'))
        return res

    def _register_operators(self) -> dict:
        """【终极防弹版】注册全部算子的 GPU Tensor 实现"""
        
        # 🛡️ 辅助函数 1：自动将标量(数字)转为与环境对齐的 Tensor
        def _t(v):
            if isinstance(v, (int, float)):
                # 如果是数字，生成一个与股票池形状一致的常数矩阵
                dummy = list(self.stock_tensor.values())[0]
                return torch.full_like(dummy, float(v), device=device)
            return v

        # 🛡️ 辅助函数 2：保证时序截断后，必定在头部填充 d-1 个 NaN (解决 1444 vs 1453 报错)
        def pad_unfold(x, d, func):
            res = torch.full_like(x, float('nan'))
            if x.shape[0] >= d:
                res[d-1:] = func(x.unfold(0, d, 1))
            return res

        def ts_cov(x, y, d):
            res = torch.full_like(x, float('nan'))
            if x.shape[0] >= d:
                unf_x = x.unfold(0, d, 1); unf_y = y.unfold(0, d, 1)
                mean_x = unf_x.nanmean(dim=2, keepdim=True)
                mean_y = unf_y.nanmean(dim=2, keepdim=True)
                res[d-1:] = ((unf_x - mean_x) * (unf_y - mean_y)).nanmean(dim=2)
            return res
            
        def ts_corr(x, y, d):
            res = torch.full_like(x, float('nan'))
            if x.shape[0] >= d:
                unf_x = x.unfold(0, d, 1); unf_y = y.unfold(0, d, 1)
                mean_x = unf_x.nanmean(dim=2, keepdim=True)
                mean_y = unf_y.nanmean(dim=2, keepdim=True)
                cov = ((unf_x - mean_x) * (unf_y - mean_y)).nanmean(dim=2)
                var_x = ((unf_x - mean_x)**2).nanmean(dim=2)
                var_y = ((unf_y - mean_y)**2).nanmean(dim=2)
                res[d-1:] = cov / torch.sqrt(var_x * var_y).clamp(min=1e-8)
            return res

        def ts_moments(x, d):
            var_res = torch.full_like(x, float('nan'))
            skew_res = torch.full_like(x, float('nan'))
            kurt_res = torch.full_like(x, float('nan'))
            if x.shape[0] >= d:
                unf = x.unfold(0, d, 1)
                valid = ~torch.isnan(unf)
                count = valid.sum(dim=2).float()
                mean = unf.nanmean(dim=2, keepdim=True)
                diff = torch.where(valid, unf - mean, torch.tensor(0.0, device=device))
                
                var = (diff**2).sum(dim=2) / (count - 1).clamp(min=1)
                std = torch.sqrt(var)
                skew = (diff**3).sum(dim=2) / count.clamp(min=1) / (std**3).clamp(min=1e-8)
                kurt = (diff**4).sum(dim=2) / count.clamp(min=1) / (var**2).clamp(min=1e-8)
                
                var_res[d-1:] = torch.where(count > 1, var, float('nan'))
                skew_res[d-1:] = torch.where(count > 2, skew, float('nan'))
                kurt_res[d-1:] = torch.where(count > 3, kurt, float('nan'))
            return var_res, torch.sqrt(var_res), skew_res, kurt_res

        def ts_slope_rsquare_resi(y, d):
            slope_res = torch.full_like(y, float('nan'))
            r2_res = torch.full_like(y, float('nan'))
            resi_res = torch.full_like(y, float('nan'))
            if y.shape[0] >= d:
                unf_y = y.unfold(0, d, 1)
                valid = ~torch.isnan(unf_y).any(dim=2) 
                
                x = torch.arange(d, dtype=torch.float32, device=device)
                x_mean = (d - 1) / 2.0
                x_var = (d**2 - 1) / 12.0
                
                y_mean = unf_y.mean(dim=2, keepdim=True)
                y_var = ((unf_y - y_mean)**2).mean(dim=2)
                cov_xy = ((unf_y - y_mean) * (x - x_mean).view(1, 1, d)).mean(dim=2)
                
                slope = cov_xy / x_var
                r2 = (cov_xy**2) / (x_var * y_var).clamp(min=1e-8)
                resi = unf_y[:, :, -1] - (y_mean.squeeze(-1) + slope * (d - 1 - x_mean))
                
                slope_res[d-1:] = torch.where(valid, slope, float('nan'))
                r2_res[d-1:] = torch.where(valid, r2, float('nan'))
                resi_res[d-1:] = torch.where(valid, resi, float('nan'))
            return slope_res, r2_res, resi_res

        def ts_wma(y, d):
            res = torch.full_like(y, float('nan'))
            if y.shape[0] >= d:
                unf = y.unfold(0, d, 1)
                w = torch.arange(1, d+1, dtype=torch.float32, device=device)
                valid = ~torch.isnan(unf)
                w_sum = torch.where(valid, w, torch.tensor(0.0, device=device)).sum(dim=2)
                w_prod = torch.where(valid, unf * w, torch.tensor(0.0, device=device)).sum(dim=2)
                res[d-1:] = torch.where(w_sum > 0, w_prod / w_sum, float('nan'))
            return res

        def ts_ema(y, d):
            res = torch.empty_like(y)
            alpha = 2.0 / (d + 1)
            curr = y[0].clone()
            for i in range(y.shape[0]):
                valid = ~torch.isnan(y[i])
                curr = torch.where(valid, torch.where(torch.isnan(curr), y[i], curr * (1 - alpha) + y[i] * alpha), curr)
                res[i] = curr
            return res

        def delay(x, d):
            res = torch.full_like(x, float('nan'))
            res[d:] = x[:-d]
            return res

        return {
            # 基础算术 (自动强制转换常量)
            'Add': lambda x, y: _t(x) + _t(y),
            'Sub': lambda x, y: _t(x) - _t(y),
            'Mul': lambda x, y: _t(x) * _t(y),
            'Div': lambda x, y: _t(x) / _t(y).clone().masked_fill_(_t(y)==0, float('nan')),
            'Neg': lambda x: -_t(x),
            'Abs': lambda x: _t(x).abs(),
            'Sign': lambda x: _t(x).sign(),                 
            'Max2': lambda x, y: torch.maximum(_t(x), _t(y)),      
            'Min2': lambda x, y: torch.minimum(_t(x), _t(y)), 
            'Log': lambda x: torch.log(_t(x).abs().clamp(min=1e-8)),
            'SignedPower': lambda x, e: _t(x).sign() * (_t(x).abs() ** e),
            'Power': lambda x, e: _t(x) ** e,
            'Sqrt': lambda x: torch.sqrt(_t(x).abs()),
            
            # 时序统计 (带 NaN 对齐护城河)
            'Mean': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.nanmean(dim=2)),
            'Std': lambda x, d: ts_moments(_t(x), d)[1],
            'Var': lambda x, d: ts_moments(_t(x), d)[0],
            'Skew': lambda x, d: ts_moments(_t(x), d)[2],
            'Kurt': lambda x, d: ts_moments(_t(x), d)[3],
            'Med': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.nanmedian(dim=2).values),
            'Sum': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.nansum(dim=2)),
            'Product': lambda x, d: pad_unfold(_t(x), d, lambda unf: torch.exp(torch.log(unf.abs().clamp(min=1e-8)).nansum(dim=2)) * (unf.sign().prod(dim=2))),

            # 时序进阶
            'Delay': lambda x, d: delay(_t(x), d),
            'Delta': lambda x, d: _t(x) - delay(_t(x), d),
            'TsRank': lambda x, d: self._ts_rank(_t(x), d),
            'TsMax': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.max(dim=2).values),
            'TsMin': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.min(dim=2).values),
            'Max': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.max(dim=2).values),
            'Min': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.min(dim=2).values),
            'TsArgMax': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.argmax(dim=2).float()),
            'TsArgMin': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.argmin(dim=2).float()),
            'TsDecay': lambda x, d: ts_wma(_t(x), d),
            
            'Cov': lambda x, y, d: ts_cov(_t(x), _t(y), d),
            'Corr': lambda x, y, d: ts_corr(_t(x), _t(y), d),
            
            # 截面与标度
            'CsRank': lambda x: self._cs_rank(_t(x)),
            'Scale': lambda x, a=1: _t(x) / _t(x).abs().sum(dim=1, keepdim=True).clamp(min=1e-8) * a,

            # 平滑与回归
            'SMA': lambda x, d: pad_unfold(_t(x), d, lambda unf: unf.nanmean(dim=2)),
            'EMA': lambda x, d: ts_ema(_t(x), d),
            'WMA': lambda x, d: ts_wma(_t(x), d),
            'Slope': lambda x, d: ts_slope_rsquare_resi(_t(x), d)[0],
            'Rsquare': lambda x, d: ts_slope_rsquare_resi(_t(x), d)[1],
            'Resi': lambda x, d: ts_slope_rsquare_resi(_t(x), d)[2],

            # 逻辑运算
            'IfElse': lambda cond, x, y: torch.where(_t(cond) > 0, _t(x), _t(y)),
            'Greater': lambda x, y: (_t(x) > _t(y)).float(),
            'Less': lambda x, y: (_t(x) < _t(y)).float(),
            'GreaterEqual': lambda x, y: (_t(x) >= _t(y)).float(),
            'LessEqual': lambda x, y: (_t(x) <= _t(y)).float(),
            'And': lambda x, y: (_t(x).bool() & _t(y).bool()).float(),
            'Or': lambda x, y: (_t(x).bool() | _t(y).bool()).float(),
            'Eq': lambda x, y: (_t(x) == _t(y)).float(),
            'Ne': lambda x, y: (_t(x) != _t(y)).float()
        }

    def _calculate_factor(self, formula: str):
        """执行表达式树，在全量时间轴上用 GPU 计算因子"""
        return eval(formula, {"__builtins__": {}}, self.safe_env)

    def _spearman_ic(self, factor_rank, target_tensor):
        """利用矩阵代数极速计算截面 Spearman 相关性"""
        target_rank = self._cs_rank(target_tensor)
        f_mean = factor_rank.nanmean(dim=1, keepdim=True)
        t_mean = target_rank.nanmean(dim=1, keepdim=True)
        cov = ((factor_rank - f_mean) * (target_rank - t_mean)).nanmean(dim=1)
        std_f = ((factor_rank - f_mean)**2).nanmean(dim=1).sqrt()
        std_t = ((target_rank - t_mean)**2).nanmean(dim=1).sqrt()
        return cov / (std_f * std_t).clamp(min=1e-8)

# ==========================================
    # 🏎️ 全流程 GPU 极速评测流水线 (带历史数据对齐保护)
    # ==========================================
    def evaluate(self, formula: str, start_date: str = None, end_date: str = None) -> tuple[bool, dict, pd.DataFrame]:
        try:
            # 1. 在全局数据上极速计算因子张量
            factor_tensor_full = self._calculate_factor(formula)
            
            # 💡[核心过滤机制]：应用股票池掩码，非池内股票因子值强制变 NaN！
            if self.pool_mask_tensor is not None:
                factor_tensor_full = torch.where(self.pool_mask_tensor, factor_tensor_full, torch.tensor(float('nan'), device=device))
                
            # 2. 定位切片索引
            slice_mask = pd.Series(True, index=self.index_dates)
            if start_date: slice_mask &= (self.index_dates >= pd.to_datetime(start_date))
            if end_date: slice_mask &= (self.index_dates <= pd.to_datetime(end_date))
            
            valid_indices = np.where(slice_mask)[0]
            if len(valid_indices) == 0:
                raise ValueError(f"选定的时间段内没有有效数据！")
            
            start_i, end_i = valid_indices[0], valid_indices[-1] + 1
            
            # 🔑 提取当前切片的标准时间轴（用于修复历史因子长度不匹配）
            target_index = self.index_dates[start_i:end_i]
            
            # 3. 截取对应区间的张量
            f_tensor = factor_tensor_full[start_i:end_i]
            fwd_ret = self.fwd_tensor[start_i:end_i]
            bench_ret = self.bench_tensor[start_i:end_i]
            
            # A. 预测能力指标 (IC 相关)
            f_rank = self._cs_rank(f_tensor)
            ic_series = self._spearman_ic(f_rank, fwd_ret)
            ic_series = ic_series[~torch.isnan(ic_series)]
            
            if len(ic_series) < 2: raise ValueError("选定时间段内的有效 IC 样本量不足")

            ic_mean = ic_series.mean().item()
            ic_std = ic_series.std().item()
            ir = ic_mean / ic_std if ic_std != 0 else 0.0
            t_value = ic_mean / (ic_std / np.sqrt(len(ic_series))) if ic_std != 0 else 0.0
            ic_win_rate = (ic_series > 0).float().mean().item() if ic_mean > 0 else (ic_series < 0).float().mean().item()
            
            # 因子衰减
            ic_decay_2 = self._spearman_ic(f_rank, self.fwd_tensor_shift_1[start_i:end_i]).nanmean().item()
            ic_decay_3 = self._spearman_ic(f_rank, self.fwd_tensor_shift_2[start_i:end_i]).nanmean().item()
            decay_tuple = (round(ic_mean, 3), round(ic_decay_2, 3), round(ic_decay_3, 3))

            # B. 收益率与分组指标
            q_mean_rets =[]
            for i in range(5):
                mask = (f_rank > i * 0.2) & (f_rank <= (i + 1) * 0.2)
                q_ret = torch.where(mask, fwd_ret, torch.tensor(float('nan'), device=device)).nanmean(dim=1)
                q_mean_rets.append(q_ret)
                
            q_means = torch.tensor([q.nanmean().item() for q in q_mean_rets], device=device)
            target = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], device=device)
            monotonicity = torch.corrcoef(torch.stack([q_means, target]))[0, 1].item()
            if np.isnan(monotonicity): monotonicity = 0.0

            if ic_mean > 0:
                ls_daily_ret = q_mean_rets[4] - q_mean_rets[0] 
            else:
                ls_daily_ret = q_mean_rets[0] - q_mean_rets[4] 
                
            ls_daily_ret = torch.where(torch.isnan(ls_daily_ret), torch.tensor(0.0, device=device), ls_daily_ret)
            
            excess_daily_ret = ls_daily_ret - bench_ret
            excess_ann_ret = excess_daily_ret.mean().item() * 252
            ls_ann_ret = ls_daily_ret.mean().item() * 252
            ls_ann_vol = ls_daily_ret.std().item() * np.sqrt(252)
            ls_ann_sharpe = excess_ann_ret / ls_ann_vol if ls_ann_vol != 0 else 0.0
            
            cum_net_value = (1 + ls_daily_ret).cumprod(dim=0)
            running_max = cum_net_value.cummax(dim=0).values
            drawdowns = (cum_net_value - running_max) / running_max
            max_drawdown = drawdowns.min().item()

            # ==========================================
            # C. 落地约束与正交化约束 (🚀已修复维度崩溃 Bug)
            # ==========================================
            turnover = (f_rank[1:] - f_rank[:-1]).abs().nanmean().item()
            
            existing_factors = self.factor_base.get_all_factor_data()
            max_corr = 0.0
            for ext_df in existing_factors.values():
                # 【防弹补丁】必须使用 reindex 强制将历史因子文件的时间轴向当前 1453 行对齐，缺失补 NaN
                ext_df_aligned = ext_df.reindex(target_index)
                
                # 现在转为 Tensor，它绝对是 1453 行，完美适配 GPU
                ext_tensor = torch.tensor(ext_df_aligned.values, dtype=torch.float32, device=device)
                ext_rank = self._cs_rank(ext_tensor)
                
                f_mean = f_rank.nanmean(dim=1, keepdim=True)
                e_mean = ext_rank.nanmean(dim=1, keepdim=True)
                cov = ((f_rank - f_mean) * (ext_rank - e_mean)).nanmean(dim=1)
                std_f = ((f_rank - f_mean)**2).nanmean(dim=1).sqrt()
                std_e = ((ext_rank - e_mean)**2).nanmean(dim=1).sqrt()
                
                corr = (cov / (std_f * std_e).clamp(min=1e-8)).nanmean().item()
                if abs(corr) > max_corr: max_corr = abs(corr)

            # D. 输出封装与准入判断
            metrics = {
                "Period": f"{start_date or 'Start'} to {end_date or 'End'}",
                "IC": round(ic_mean, 4),
                "IR": round(ir, 4),
                "IC_WinRate": round(ic_win_rate, 4),
                "t_value": round(t_value, 2),
                "EXCESS_Return": round(excess_ann_ret, 4),
                "LS_Return": round(ls_ann_ret, 4),
                "LS_Sharpe": round(ls_ann_sharpe, 4),
                "Max_Drawdown": round(max_drawdown, 4),
                "Monotonicity": round(monotonicity, 4),
                "Turnover": round(turnover, 4),
                "Max_Correlation": round(max_corr, 4),
                "Decay(T1-T3)": decay_tuple
            }
            
            is_success = (
                (abs(ic_mean) > 0.03) and 
                (abs(t_value) > 2.0) and  
                (turnover < 0.3) and      
                (max_corr < 0.6)          
            )
            
            # 仅返回全量数据的 DataFrame 以便后续存储与对齐
            factor_df_full = pd.DataFrame(factor_tensor_full.cpu().numpy(), 
                                          index=self.index_dates, 
                                          columns=self.columns_stocks)
            
            return is_success, metrics, factor_df_full

        except Exception as e:
            return False, {"Error": str(e)}, None

🚀 因子评测引擎已挂载至计算节点: cuda


In [23]:
class FactorMiningAgent:
    def __init__(self, info_base: InformationBase, factor_base: FactorBase, eval_engine: EvaluationEngine, api_key: str, unlogic=True):
        self.info_base = info_base
        self.factor_base = factor_base
        self.eval_engine = eval_engine
        self.client = openai.Client(api_key=api_key,base_url="https://api.chatanywhere.tech/v1")
        self.unlogic=unlogic

    def generate(self) -> str:
        """Step 1 & 2: 读取外部记忆，生成新公式"""
        if self.unlogic:
            prompt = f"""
            你是一个顶尖的量化因子挖掘专家。请根据信息库生成一个全新的 Alpha 因子公式，请遵循信息库的‘禁忌’，参考“推荐”，但除此之外，不必受限于传统金融学常识的束缚，可以发挥数学，机器学习，统计等知识。
            
            【可用数据列表】
            Close, Open, High, Low, Volume, Amount

            【可用算子库】
            1. 算术: Add(x,y), Sub(x,y), Mul(x,y), Div(x,y), Neg(x), Abs(x), Sign(x), Max2(x,y), Min2(x,y), Log(x), SignedPower(x,e), Power(x,e), Sqrt(x)
            2. 统计(滚动d天): Mean(x,d), Std(x,d), Skew(x,d), Kurt(x,d), Med(x,d), Sum(x,d)
            3. 时序与相关(滚动d天): Delay(x,d), Delta(x,d), TsRank(x,d), TsMax(x,d), TsMin(x,d), TsArgMax(x,d), TsDecay(x,d), Cov(x,y,d), Corr(x,y,d)
            4. 截面: CsRank(x), Scale(x)
            5. 平滑: SMA(x,d), EMA(x,d), WMA(x,d)
            6. 时序回归(滚动d天): Slope(x,d), Rsquare(x,d), Resi(x,d)
            7. 逻辑(返回条件判断): IfElse(cond, x, y), Greater(x,y), Less(x,y), And(x,y), Or(x,y), Eq(x,y)

            【公式示例】
            - 影线逻辑: Div(Sub(Min2(Open, Close), Low), Add(Sub(High, Low), 0.0001))
            - 量价协方差反转: Neg(Mul(TsRank(Cov(TsRank(Close,18), TsRank(Volume,18), 10), 18), CsRank(Returns)))
            - 状态切换: IfElse(Greater(Kurt(Returns, 24), 3), Neg(Resi(Close, 6)), Neg(Slope(Close, 12)))
            【已有因子】（禁止重复）:\n{self.factor_base.get_prompt_summary()}
            【信息库经验】（必须遵循）:\n{self.info_base.get_prompt_summary()}
            
            仅输出一行纯文本的数学公式。
            """
            response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=1
        )
            return response.choices[0].message.content.strip()
        
        else:
            prompt = f"""
        你是一位顶尖的量化因子挖掘专家。你的任务是结合市场微观结构、行为金融学或传统的量价逻辑，构建一个全新的、具有强逻辑支撑的 Alpha 因子。

        【可用数据列表】
        Close, Open, High, Low, Volume, Amount

        【可用算子库】
        1. 算术: Add(x,y), Sub(x,y), Mul(x,y), Div(x,y), Neg(x), Abs(x), Sign(x), Max2(x,y), Min2(x,y), Log(x), SignedPower(x,e), Power(x,e), Sqrt(x)
        2. 统计(滚动d天): Mean(x,d), Std(x,d), Skew(x,d), Kurt(x,d), Med(x,d), Sum(x,d)
        3. 时序与相关(滚动d天): Delay(x,d), Delta(x,d), TsRank(x,d), TsMax(x,d), TsMin(x,d), TsArgMax(x,d), TsDecay(x,d), Cov(x,y,d), Corr(x,y,d)
        4. 截面处理(消除市场beta): CsRank(x), Scale(x)
        5. 平滑(降低换手): SMA(x,d), EMA(x,d), WMA(x,d)
        6. 时序回归(滚动d天): Slope(x,d), Rsquare(x,d), Resi(x,d)
        7. 逻辑(状态切换): IfElse(cond, x, y), Greater(x,y), Less(x,y), And(x,y), Or(x,y), Eq(x,y)

            【公式示例】
            - 影线逻辑: Div(Sub(Min2(Open, Close), Low), Add(Sub(High, Low), 0.0001))
            - 量价协方差反转: Neg(Mul(TsRank(Cov(TsRank(Close,18), TsRank(Volume,18), 10), 18), CsRank(Returns)))
            - 状态切换: IfElse(Greater(Kurt(Returns, 24), 3), Neg(Resi(Close, 6)), Neg(Slope(Close, 12)))
            【已有因子】（禁止重复）:\n{self.factor_base.get_prompt_summary()}
            【信息库经验】（必须遵循）:\n{self.info_base.get_prompt_summary()}
            

        【输出格式要求】
        请使用严谨的量化思维进行逐步推理，并严格返回如下 JSON 格式（不要附带 markdown 等其他字符）：
        {{
            "financial_hypothesis": "阐述该因子的金融逻辑与微观原理。",
            "operator_mapping": "说明为什么选择某些具体的算子组合来实现上述假说，特别是如何处理量纲、极端值和降低换手率。",
            "formula": "最终的一行纯文本数学公式（仅包含公式本身，禁止任何换行或额外字符）。"
        }}
        """
        raw_content = ""
        try:
            response = self.client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                temperature=1  # 挖掘阶段保持一定的创造力
            )
            raw_content = response.choices[0].message.content.strip()
            
            # 【核心：强力 JSON 清洗防御】
            cleaned_content = raw_content.replace('```json', '').replace('```', '').strip()
            
            # 用正则暴力提取大括号内的内容
            match = re.search(r'\{.*\}', cleaned_content, re.DOTALL)
            if match:
                cleaned_content = match.group(0)
                
            # 去除可能导致解析崩溃的非法换行/制表符，但保留公式的连贯性
            cleaned_content = cleaned_content.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            # 解析 JSON
            parsed_data = json.loads(cleaned_content)
            
            # 提取字段
            formula = parsed_data.get("formula", "").strip()
            hypothesis = parsed_data.get("financial_hypothesis", "未提供逻辑假说")
            mapping = parsed_data.get("operator_mapping", "未提供算子映射")
            
            if not formula:
                raise ValueError("LLM 未在 JSON 中返回 formula 字段")
                
            return formula
            
        except json.JSONDecodeError:
            print(f"\n⚠️ [生成跳过] LLM 输出 JSON 格式异常。原始输出片段: {raw_content[:150]}...")
            return None, None, None
        except Exception as e:
            print(f"\n⚠️ [生成跳过] 因子生成阶段发生异常: {e}")
            return None, None, None

    def distill(self, formula: str, metrics: dict, is_success: bool):
        """Step 4: 基于评测结果进行反思，并更新信息库"""
        status = "成功" if is_success else "失败"

        prompt = f"""
        你是一位资深且严谨的量化评测总监。你正在对一个新挖掘的 Alpha 因子进行回测结果的回顾与反思。
        
        【测试目标】
        公式: {formula}
        回测结果状态: {status}
        详细指标 : {json.dumps(metrics)}
        
        【任务要求】
        请基于以上量化指标进行深度的归因分析，并提取一条可泛化的经验/规则，更新到因子挖掘信息库中。
        1. 若成功：分析是哪种金融逻辑（如动量反转、波动率调整、量价背离等）或算子组合起到了关键作用。
        2. 若失败：诊断失败的根本原因（如：换手率过高导致摩擦成本大、过度拟合、线性相关性太高、算子量纲不匹配等）。
        
        【输出格式要求】
        严格输出一个合法的 JSON 对象，不要包含任何 markdown 语法（不要 ```json），不要换行控制符，不要解释！反思的action严格参照是否成功！格式如下：
        {{
            "action": "add_recommended" 或者 "add_prohibited",
            "content": "提炼出的一句高度精炼的量化经验法则（例如：'避免对高频量价相关性直接求和，需先进行截面去均值处理'）。"
        }}
        """
        try:
            response = self.client.chat.completions.create(
                model="gpt-5-mini",  
                messages=[{"role": "user", "content": prompt}],
                temperature=1
            )
            raw_content = response.choices[0].message.content.strip()
            
            # 【清洗防御 1】：去除 Markdown 标记
            cleaned_content = raw_content.replace('```json', '').replace('```', '').strip()
            
            # 【清洗防御 2】：使用正则表达式强行提取大括号 {} 里的内容，防止 LLM 前后说废话
            match = re.search(r'\{.*\}', cleaned_content, re.DOTALL)
            if match:
                cleaned_content = match.group(0)
            
            # 【清洗防御 3】：去除可能导致 control character 报错的非法换行/制表符
            cleaned_content = cleaned_content.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            # 尝试解析
            instruction = json.loads(cleaned_content)
            
            action = instruction.get("action")
            content = instruction.get("content")
            
            if action and content:
                self.info_base.update_experience(action, content)
                # print(f"  [反思记录] {action}: {content}") # 如果需要看反思内容可以取消注释
                
        except json.JSONDecodeError as e:
            # 如果解析依然失败，打印错误，但使用 pass 跳过，保证循环继续
            print(f"\n⚠️ [解析跳过] LLM 输出格式异常无法解析。原始输出: {raw_content}")
        except Exception as e:
            # 捕获其他任何网络断开或 API 错误
            print(f"\n⚠️ [网络/API 异常] 反思阶段出错: {e}")

    def run_ralph_loop(self, max_iterations: int = 100, start_date='2015-01-01', end_date='2020-12-31'):
        """核心无状态迭代循环"""
        for i in range(max_iterations):
            print(f"\n--- 启动第 {i+1} 轮迭代 ---")
            
            # 1. 生成因子
            formula = self.generate()
            print(f"Agent 生成公式: {formula}")
            
            # 2. 客观评测
            is_success, metrics, factor_df = self.eval_engine.evaluate(formula, start_date=start_date, end_date=end_date)
            print(f"评测结果: 是否成功={is_success}, 指标={metrics}")
            
            # 3. 提炼反思
            self.distill(formula, metrics, is_success)
            
            # 4. 入库结算
            if is_success and factor_df is not None:
                self.factor_base.add_factor(formula, metrics, factor_df)
                print(f"✅ 因子入库成功！ID 记录已更新。")
                
            time.sleep(1) # 避免 API 限流

In [22]:
class TushareDataLoader:
    def __init__(self, token: str, start_date: str, end_date: str, stock_id=None, cache_dir: str = "./data_cache/"):
        self.token = token
        self.start_date = start_date
        self.end_date = end_date
        self.cache_dir = cache_dir
        
        # 兼容不同类型的 stock_id 传入格式
        if stock_id is None:
            self.stockid =[]
        elif isinstance(stock_id, str):
            self.stockid = [stock_id]
        else:
            self.stockid = stock_id

        ts.set_token(self.token)
        self.pro = ts.pro_api()
        os.makedirs(self.cache_dir, exist_ok=True)

    def fetch_daily_data(self) -> tuple:
        """
        按交易日循环获取全市场日线数据及复权因子，并生成【未复权】与【后复权】双路宽表矩阵。
        :return: (unadj_data: dict, post_adj_data: dict)
        """
        # 构建带特征哈希的缓存文件名，防止不同股票池缓存冲突
        if self.stockid:
            code_ident = hashlib.md5("_".join(sorted(self.stockid)).encode()).hexdigest()[:8]
            cache_file = os.path.join(self.cache_dir, f"custom_{code_ident}_{self.start_date}_{self.end_date}.parquet")
        else:
            cache_file = os.path.join(self.cache_dir, f"all_market_{self.start_date}_{self.end_date}.parquet")
        
        # ==========================================
        # 第一步：缓存读取 或 交易日历循环拉取
        # ==========================================
        if os.path.exists(cache_file):
            print(f"命中本地缓存: {cache_file}，正在使用 PyArrow 飞速加载全市场数据...")
            all_data = pd.read_parquet(cache_file)
        else:
            print(f"未命中缓存，正在获取 {self.start_date} 到 {self.end_date} 的交易日历...")
            # 获取区间内所有的交易日
            cal_df = self.pro.trade_cal(exchange='SSE', start_date=self.start_date, end_date=self.end_date, is_open=1)
            trade_dates = cal_df['cal_date'].tolist()
            
            if not trade_dates:
                raise ValueError(f"在 {self.start_date} 到 {self.end_date} 期间，未找到有效交易日！")

            print(f"共获取到 {len(trade_dates)} 个交易日，开始按日循环拉取量价与复权因子...")
            all_data_list =[]
            
            for i, date_str in enumerate(trade_dates):
                try:
                    # 1. 拉取当日所有股票的数据
                    df_daily = self.pro.daily(trade_date=date_str)
                    # 2. 拉取当日复权因子
                    df_adj = self.pro.adj_factor(trade_date=date_str)
                    
                    if not df_daily.empty and not df_adj.empty:
                        # 融合复权因子
                        df_adj = df_adj[['ts_code', 'adj_factor']]
                        df_merged = pd.merge(df_daily, df_adj, on='ts_code', how='left')
                        
                        # 如果指定了目标股票池，进行截断过滤
                        if self.stockid:
                            df_merged = df_merged[df_merged['ts_code'].isin(self.stockid)]
                            
                        all_data_list.append(df_merged)

                    # 严格控制休眠防封，按天拉取每天调用2个接口
                    time.sleep(0.12) 
                    
                    if (i + 1) % 100 == 0:
                        print(f"进度：已拉取 {i + 1} / {len(trade_dates)} 个交易日的数据...")
                        
                except Exception as e:
                    print(f"  [x] 下载 {date_str} 截面数据失败: {e}")
                    time.sleep(1) # 遇到网络波动稍微多等一会
                    
            print("\n正在拼接全市场数据表...")
            all_data = pd.concat(all_data_list, ignore_index=True)
            all_data['trade_date'] = pd.to_datetime(all_data['trade_date'])
            all_data.sort_values(['trade_date', 'ts_code'], inplace=True)
            
            all_data.to_parquet(cache_file, engine='pyarrow')
            print("全股票数据下载完成并已安全落盘 (Parquet格式)！\n")

        # ==========================================
        # 第二步：数据透视（长表转宽表矩阵）及 停牌逻辑处理
        # ==========================================
        print("正在构建量化计算引擎数据矩阵 (进行矩阵透视及停牌清洗)...")
        pivot_dict = {}
        cols_to_pivot =['open', 'high', 'low', 'close', 'vol', 'amount', 'adj_factor']
        
        for col in cols_to_pivot:
            if col not in all_data.columns: continue
            pivot_df = all_data.pivot(index='trade_date', columns='ts_code', values=col)
            pivot_df.sort_index(inplace=True)
            pivot_dict[col] = pivot_df

        # 重点逻辑：区分特征进行不同的缺失值(停牌)填充
        # 1. 价格和复权因子类 -> 停牌沿用昨收（向下填充 ffill）
        price_cols =['open', 'high', 'low', 'close', 'adj_factor']
        for col in price_cols:
            if col in pivot_dict:
                pivot_dict[col] = pivot_dict[col].ffill()
                
        # 2. 交易热度类 -> 停牌真实成交量/成交额为 0 (填充 0)
        vol_cols = ['vol', 'amount']
        for col in vol_cols:
            if col in pivot_dict:
                pivot_dict[col] = pivot_dict[col].fillna(0)

        # ==========================================
        # 第三步：计算双通道（复权与未复权）数据字典
        # ==========================================
        unadj_data = {}
        post_adj_data = {}
        
        # 组装未复权字典
        unadj_data['Open']       = pivot_dict.get('open')
        unadj_data['High']       = pivot_dict.get('high')
        unadj_data['Low']        = pivot_dict.get('low')
        unadj_data['Close']      = pivot_dict.get('close')
        unadj_data['Volume']     = pivot_dict.get('vol')
        unadj_data['Amount']     = pivot_dict.get('amount')
        unadj_data['Adj_Factor'] = pivot_dict.get('adj_factor') # 保留因子备用

        # 组装后复权字典：后复权价格 = 真实价格 * 复权因子
        adj_f = pivot_dict.get('adj_factor')
        if adj_f is not None:
            post_adj_data['Open']   = pivot_dict['open'] * adj_f
            post_adj_data['High']   = pivot_dict['high'] * adj_f
            post_adj_data['Low']    = pivot_dict['low'] * adj_f
            post_adj_data['Close']  = pivot_dict['close'] * adj_f
            post_adj_data['Volume'] = pivot_dict['vol'] 
            post_adj_data['Amount'] = pivot_dict['amount'] * adj_f     # 真实资金流水不需要复权
        
        return unadj_data, post_adj_data

    def build_forward_returns(self, close_df: pd.DataFrame, periods: int = 1) -> pd.DataFrame:
        """
        构建未来收益率矩阵 (用于 IC 测试)
        【注意】：参数 close_df 必须传入 `post_adj_data['Close']` (后复权收盘价)，
        否则分红除权日收益率会出现异常断崖。
        """
        forward_returns = close_df.shift(-periods) / close_df - 1.0
        return forward_returns
    
    def fetch_benchmark_returns(self, ts_code='000300.SH') -> pd.Series:
        """获取沪深300作为基准的标准每日收益率 (T日真实收益)"""
        print(f"正在拉取基准指数 {ts_code} 数据...")
        
        df_index = self.pro.index_daily(ts_code=ts_code, start_date=self.start_date, end_date=self.end_date)
        if df_index is None or df_index.empty:
             raise ValueError("获取指数基准数据失败，请检查网络或权限！")
             
        df_index['trade_date'] = pd.to_datetime(df_index['trade_date'])
        df_index.sort_values('trade_date', inplace=True)
        df_index.set_index('trade_date', inplace=True)
        
        # 💡 [修复] 恢复为标准的每日真实收益率 (T日收盘 / T-1日收盘 - 1)
        # 不再在这里 shift(-1)，还原大盘真实走势！
        bench_ret = df_index['close'].pct_change().fillna(0)
        
        return bench_ret
    
    def fetch_dynamic_pool_mask(self, pool_code: str, target_index: pd.DatetimeIndex, target_columns: pd.Index) -> pd.DataFrame:
        """
        获取指定指数的【实时动态成分股掩码矩阵】(Time x Stocks)
        True 代表该股票当日在池子内，False 代表不在。
        
        :param pool_code: 例如 '000300.SH' (沪深300), '000905.SH' (中证500)
        """
        if not pool_code:
            # 如果不给定池子，返回全 True 的掩码（全市场）
            return pd.DataFrame(True, index=target_index, columns=target_columns)
            
        print(f"🌐 正在构建 {pool_code} 历史动态成分股掩码矩阵...")
        
        # 提取年份，按年循环获取以防 Tushare 单次 5000 条限制报错
        start_year = target_index.min().year
        end_year = target_index.max().year
        
        df_list =[]
        for y in range(start_year, end_year + 1):
            try:
                df = self.pro.index_weight(index_code=pool_code, start_date=f"{y}0101", end_date=f"{y}1231")
                df_list.append(df)
                time.sleep(0.15)
            except Exception as e:
                print(f"获取 {y} 年成分股失败: {e}")
                
        if not df_list:
            raise ValueError(f"无法获取 {pool_code} 的成分股数据，请检查权限。")
            
        weight_df = pd.concat(df_list, ignore_index=True)
        weight_df['trade_date'] = pd.to_datetime(weight_df['trade_date'])
        
        # 1. 转换为宽表 (有权重的代表在池内，NaN代表不在)
        mask_wide = weight_df.pivot(index='trade_date', columns='con_code', values='weight')
        mask_wide = ~mask_wide.isna() # 转换为 Boolean 矩阵
        
        # 2. 对齐目标时间轴：指数权重通常是月度更新，我们需要前向填充 (ffill) 到每一天
        mask_wide = mask_wide.reindex(index=target_index).ffill().fillna(False)
        
        # 3. 对齐目标股票轴：全市场对齐，缺失的列填充 False
        mask_wide = mask_wide.reindex(columns=target_columns, fill_value=False)
        
        print(f"✅ 动态成分股矩阵构建完成！")
        return mask_wide

In [20]:
# ==========================================
# 1. 独立的 Backtrader 组件区 (避免嵌套类引发的序列化问题)
# ==========================================
class FactorData(bt.feeds.PandasData):
    lines = ('factor',)
    params = (('datetime', None), ('open', 'open'), ('high', 'high'), 
              ('low', 'low'), ('close', 'close'), ('volume', 'vol'), ('factor', 'factor'))

class TurnoverAndFeeAnalyzer(bt.Analyzer):
    def __init__(self):
        self.total_commission = 0.0      
        self.total_traded_value = 0.0    
        self.daily_turnover = {}         
        self.current_day_traded_value = 0.0

    def notify_order(self, order):
        if order.status in [order.Completed]:
            self.total_commission += order.executed.comm
            trade_val = abs(order.executed.size) * order.executed.price
            self.total_traded_value += trade_val
            self.current_day_traded_value += trade_val

    def next(self):
        port_value = self.strategy.broker.getvalue()
        if port_value > 0:
            t_rate = self.current_day_traded_value / port_value
            self.daily_turnover[self.strategy.datetime.date()] = t_rate
        self.current_day_traded_value = 0.0

    def get_analysis(self):
        days_traded = len(self.daily_turnover)
        avg_daily_turnover = sum(self.daily_turnover.values()) / days_traded if days_traded > 0 else 0.0
        return {
            'Total_Commission': self.total_commission,
            'Total_Traded_Value': self.total_traded_value,
            'Average_Daily_Turnover': avg_daily_turnover,
            'Annual_Turnover': avg_daily_turnover * 252,
            'Daily_Turnover_Series': self.daily_turnover
        }

class CrossSectionalFactorStrategy(bt.Strategy):
    # 增加了一个参数 log_file，默认输出为 trade_log.txt
    params = (('top_k', 10), ('rebalance_days', 1), ('log_file', 'trade_log.txt'))

    def __init__(self):
        self.days_passed = 0
        
        # ======================================================
        # 📝 1. 初始化时打开 TXT 文件并写入表头
        # ======================================================
        self.f_log = open(self.p.log_file, 'w', encoding='utf-8')
        self.f_log.write(f" {'交易日期':^10} | {'交易方向':^10} | {'股票代码':^9} | {'成交均价':^8} | {'成交数量':^8} | {'手续费':^8}\n")
        self.f_log.write("-" * 75 + "\n")

    def log(self, txt, dt=None):
        """基础日志写入函数"""
        dt = dt or self.datetime.date()
        self.f_log.write(f"[{dt}] {txt}\n")

    # ======================================================
    # 📝 2. 监听每一笔订单的执行结果
    # ======================================================
    def notify_order(self, order):
        # 如果订单处于提交或接受状态，不打印（避免日志太啰嗦）
        if order.status in [order.Submitted, order.Accepted]:
            return

        # 如果订单已成功撮合成交
        if order.status in [order.Completed]:
            # 判断是买入还是卖出
            action = '买入 (BUY) ' if order.isbuy() else '卖出 (SELL)'
            ticker = order.data._name
            price = order.executed.price
            size = order.executed.size
            comm = order.executed.comm

            # 格式化输出 (保持每列对齐，强迫症福音)
            log_msg = f" {action:^10} | {ticker:^9} | ¥{price:>7.2f} | {size:>8} 股 | ¥{comm:.4f}"
            self.log(log_msg)

        # 如果订单因为资金不足被拒，或者被撤销
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f" ⚠️ 订单失败 (资金不足/停牌) | 股票: {order.data._name}")

    def prenext(self):
        self.next()

    def next(self):
        self.days_passed += 1
        if self.days_passed % self.p.rebalance_days != 0: return

        valid_stocks =[d for d in self.datas if len(d) > 0 and not np.isnan(d.factor[0])]
        if not valid_stocks: return

        valid_stocks.sort(key=lambda d: d.factor[0], reverse=True)
        top_k_stocks = valid_stocks[:self.p.top_k]
        
        # 预留万分之五的现金防 Margin Rejected
        target_weight = 0.995 / self.p.top_k 
        
        for d in self.datas:
            if self.getposition(d).size > 0 and d not in top_k_stocks:
                self.order_target_percent(d, target=0.0)
                
        for d in top_k_stocks:
            self.order_target_percent(d, target=target_weight)

    # ======================================================
    # 📝 3. 回测结束时安全关闭文件
    # ======================================================
    def stop(self):
        if not self.f_log.closed:
            self.f_log.write("-" * 75 + "\n")
            self.f_log.write(f" 回测结束，最终资金: ¥{self.broker.getvalue():.2f}\n")
            self.f_log.close()


# ==========================================
# 2. 核心回测引擎类封装
# ==========================================
class FactorBacktestEngine:
    """量化多头因子回测引擎"""
    
    def __init__(self, market_data_path: str, local_factor_dir: str = "all_market_factor_data"):
        """
        :param market_data_path: 全市场行情数据表路径 (Parquet格式)
        :param local_factor_dir: 本地因子库存储目录
        """
        self.market_data_path = market_data_path
        self.local_factor_dir = local_factor_dir
        
        # 确保本地因子文件夹存在
        os.makedirs(self.local_factor_dir, exist_ok=True)

    def run_backtest(self, 
                     start_date: str, 
                     end_date: str,
                     data_loader=None,
                     bench_code: str = '000300.SH', #基准指数代码
                     pool_code: str = None,    #股票池掩码   
                     factor_mode: str = 'formula', 
                     formula: str = None, 
                     factor_name: str = None, 
                     top_k: int = 10, 
                     rebalance_days: int = 1, 
                     max_stocks_in_bt: int = 200, 
                     initial_cash: float = 1000000.0, 
                     commission: float = 0.0015,
                     trade_log_file: str = "My_Trade_Log.txt"):
        """
        执行因子回测流水线
        :param factor_mode: 'formula' (公式实时计算) 或 'local' (读取本地 Parquet)
        """
        print(f"\n" + "="*50)
        print(f"🚀 初始化回测引擎 | 模式: {factor_mode.upper()} | {start_date} -> {end_date}")
        print("="*50)

        # ---------------------------------------------------------
        # 1. 加载并切片底层行情数据
        # ---------------------------------------------------------
        print(f"📥 1. 加载本地行情库: {self.market_data_path}")
        all_data = pd.read_parquet(self.market_data_path)
        all_data['trade_date'] = pd.to_datetime(all_data['trade_date'])
        
        mask = (all_data['trade_date'] >= pd.to_datetime(start_date)) & \
               (all_data['trade_date'] <= pd.to_datetime(end_date))
        sliced_data = all_data.loc[mask].copy()
        
        if sliced_data.empty:
            raise ValueError("切片区间内没有行情数据，请检查日期！")

        # 构造基准收益率 (全市场等权)
        dummy_bench = data_loader.fetch_benchmark_returns(ts_code=bench_code)
        
        if pool_code and data_loader:
            pool_mask = data_loader.fetch_dynamic_pool_mask(
                pool_code=pool_code, 
                target_index=sliced_data['trade_date'].unique(), 
                target_columns=sliced_data['ts_code'].unique()
            )
        else:
            pool_mask = None
            print("🌐 未指定动态股票池，默认在【全市场】范围内进行回测...")
        # ---------------------------------------------------------
        # 2. 获取因子数据矩阵
        # ---------------------------------------------------------
        if factor_mode == 'formula':
            if not formula: raise ValueError("Formula 模式必须提供 formula 参数！")
            factor_df_full = self._calculate_formula_factor(sliced_data, formula, dummy_bench)
            
        elif factor_mode == 'local':
            if not factor_name: raise ValueError("Local 模式必须提供 factor_name 参数！")
            factor_df_full = self._load_local_factor(factor_name)
            
        else:
            raise ValueError("factor_mode 只能为 'formula' 或 'local'")

        # ---------------------------------------------------------
        # 3. 对齐数据并拼接宽表转长表
        # ---------------------------------------------------------
        print("🔄 3. 正在将因子矩阵对齐并合并回长表...")
        slice_mask = (factor_df_full.index >= pd.to_datetime(start_date)) & \
                     (factor_df_full.index <= pd.to_datetime(end_date))
        factor_df_wide = factor_df_full.loc[slice_mask]
        del factor_df_full # 清理内存
        
        if pool_mask is not None:
            # 确保对齐
            pool_mask_aligned = pool_mask.reindex(index=factor_df_wide.index, columns=factor_df_wide.columns, fill_value=False)
            factor_df_wide = factor_df_wide.where(pool_mask_aligned, np.nan)
        
        if factor_df_wide.empty: raise ValueError("切片区间内没有因子数据！")
            
        factor_long = factor_df_wide.unstack().reset_index()
        factor_long.columns = ['ts_code', 'trade_date', 'factor']
        del factor_df_wide
        
        merged_data = pd.merge(sliced_data, factor_long, on=['trade_date', 'ts_code'], how='inner')
        merged_data.set_index('trade_date', inplace=True)
        merged_data.sort_index(inplace=True)
        del sliced_data, factor_long

        # ---------------------------------------------------------
        # 4. 构建 Backtrader 回测环境
        # ---------------------------------------------------------
        print(f"🛡️ 4. 提取流动性最好的前 {max_stocks_in_bt} 只股票以防内存溢出...")
        amount_sum = merged_data.groupby('ts_code')['amount'].sum()
        top_liquidity_stocks = amount_sum.nlargest(max_stocks_in_bt).index.tolist()
        
        print(f"🚀 5. 启动 Backtrader (TopK={top_k}, 调仓期={rebalance_days}天)...")
        cerebro = bt.Cerebro()
        for ticker in top_liquidity_stocks:
            ticker_data = merged_data[merged_data['ts_code'] == ticker].copy()
            if len(ticker_data) < 20: continue 
            cerebro.adddata(FactorData(dataname=ticker_data), name=ticker)

        cerebro.addstrategy(CrossSectionalFactorStrategy, top_k=top_k, rebalance_days=rebalance_days,log_file=trade_log_file)
        cerebro.broker.setcash(initial_cash)
        cerebro.broker.setcommission(commission=commission) 
        
        cerebro.addanalyzer(TurnoverAndFeeAnalyzer, _name='turnover_fee')
        cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='returns')

        print(f"💰 初始资金: {cerebro.broker.getvalue():.2f}")
        results = cerebro.run()
        print(f"💰 最终资金: {cerebro.broker.getvalue():.2f}")

        # ---------------------------------------------------------
        # 5. 分析指标与内存清理
        # ---------------------------------------------------------
        strat = results[0]
        tf_analysis = strat.analyzers.turnover_fee.get_analysis()
        daily_returns = pd.Series(strat.analyzers.returns.get_analysis())
        
        # 核弹级垃圾回收
        del cerebro, results, strat, merged_data, amount_sum
        gc.collect()

        self._print_trade_stats(tf_analysis, initial_cash)

        # ---------------------------------------------------------
        # 6. 生成 QuantStats HTML 报告
        # ---------------------------------------------------------
        print("📊 6. 正在后台生成 QuantStats 评估报告...")
        rpt_title = factor_name if factor_mode == 'local' else "AI_Formula_Alpha"
        report_name = f'Report_{rpt_title}_{start_date}_to_{end_date}.html'
        
        self._generate_qs_report(daily_returns, dummy_bench, rpt_title, report_name)
        
        return daily_returns, tf_analysis

    # ========================== 内部辅助方法 ==========================

    def _calculate_formula_factor(self, sliced_data, formula, dummy_bench):
        print("⚙️ 2. 准备宽表并调用 GPU 计算引擎...")
        stock_data_wide = {}
        for f in['open', 'high', 'low', 'close', 'vol', 'amount']:
            pivot_df = sliced_data.pivot(index='trade_date', columns='ts_code', values=f)
            key_name = 'Volume' if f == 'vol' else f.capitalize()
            stock_data_wide[key_name] = pivot_df.ffill()

        # 注意：这里需要依赖你定义好的 EvaluationEngine
        engine_back = EvaluationEngine(
            stock_data=stock_data_wide, 
            forward_returns=dummy_bench.to_frame(), 
            factor_base=None, benchmark_returns=dummy_bench
        )
        factor_tensor = engine_back._calculate_factor(formula)
        
        df = pd.DataFrame(
            factor_tensor.cpu().numpy() if hasattr(factor_tensor, 'cpu') else factor_tensor, 
            index=stock_data_wide['Close'].index,       # 修改这里
            columns=stock_data_wide['Close'].columns 
        )
        del stock_data_wide, engine_back
        return df

    def _load_local_factor(self, factor_name):
        local_file = os.path.join(self.local_factor_dir, f"{factor_name}.parquet")
        if not os.path.exists(local_file):
            raise FileNotFoundError(f"未找到本地因子文件: {local_file}")
            
        print(f"📥 2. 读取本地预计算因子矩阵: {local_file}")
        df = pd.read_parquet(local_file)
        df.index = pd.to_datetime(df.index)
        return df

    def _print_trade_stats(self, tf_analysis, initial_cash):
        print("\n" + "="*40)
        print("📊 交易费率与换手统计报告")
        print("="*40)
        print(f"💰 累计总手续费: {tf_analysis['Total_Commission']:,.2f} 元")
        print(f"💸 累计总交易额: {tf_analysis['Total_Traded_Value']:,.2f} 元")
        print(f"🔄 日均换手率:   {tf_analysis['Average_Daily_Turnover'] * 100:.2f} %")
        print(f"🌪️ 年化换手率:   {tf_analysis['Annual_Turnover']:,.2f} 倍")
        print(f"📉 手续费磨损率: {(tf_analysis['Total_Commission'] / initial_cash) * 100:.2f}% (占本金)")
        print("="*40)

    def _generate_qs_report(self, daily_returns, dummy_bench, title, report_name):
        daily_returns.index = pd.to_datetime(daily_returns.index).tz_localize(None).normalize()
        daily_returns = daily_returns.replace([np.inf, -np.inf], np.nan).fillna(0)
        
        dummy_bench.index = pd.to_datetime(dummy_bench.index).tz_localize(None).normalize()
        benchmark_series = dummy_bench.reindex(daily_returns.index).fillna(0)
        
        daily_returns.name = 'AI_Strategy'
        benchmark_series.name = 'Market_Avg'
        
        try:
            qs.reports.html(
                returns=daily_returns, benchmark=benchmark_series, 
                title=title, output=report_name
            )
            print(f"✅ 回测全流程完成！\n请在浏览器中打开查看报告: {os.path.abspath(report_name)}")
        except Exception as e:
            print(f"❌ QuantStats 报告生成失败: {e}")

In [ ]:
if __name__ == "__main__":
    # 配置你的 Token 和 API Key
    TUSHARE_TOKEN = "1067ed1160b1dedeca68f122160aca415393dfd50f9f49ccbd1d0580"   # 去 Tushare 官网获取
    OPENAI_API_KEY = "sk-CqqsrgnNF94QRUDkpWmuC8NvVU12ptKDGycqnGzUZUz5Jy75"  # 大模型 API Key

    print("\n[System] 启动全市场因子挖掘环境...")
    
    # 1. 实例化 Tushare 数据加载器
    data_loader = TushareDataLoader(
        token=TUSHARE_TOKEN, 
        start_date="20200316", 
        end_date="20260316"
    )
    
    # 获取全市场 5000+ 股票的宽表矩阵
    unpost_stock_data,stock_data = data_loader.fetch_daily_data()
    real_benchmark_returns = data_loader.fetch_benchmark_returns(ts_code='000300.SH')
    forward_benchmark_returns = real_benchmark_returns.shift(-1)
    # 收益率矩阵构建
    forward_returns = data_loader.build_forward_returns(stock_data['Close'], periods=1)

    print("\n[System] 初始化智能体依赖库...")
    
    # 2. 实例化记忆存储 (区分存储，避免与之前沪深300的混淆)
    info_memory = InformationBase("all_market_info_base.json")
    factor_assets = FactorBase("all_market_factor_base.json", "./all_market_factor_data/")

    # 3. 实例化评测引擎
    engine = EvaluationEngine(
        stock_data=stock_data, 
        forward_returns=forward_returns, 
        factor_base=factor_assets,
        benchmark_returns=forward_benchmark_returns
    )

    # 4. 实例化因子挖掘 Agent 
    agent = FactorMiningAgent(
        info_base=info_memory, 
        factor_base=factor_assets, 
        eval_engine=engine, 
        api_key=OPENAI_API_KEY
    )

    print("\n[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...")
    
    # 5. 启动迭代
    try:
        agent.run_ralph_loop(max_iterations=5,start_date="20200316", end_date="20250316")
    except KeyboardInterrupt:
        print("\n[System] 收到中断信号，因子挖掘系统已安全停止。")


[System] 启动全市场因子挖掘环境...
命中本地缓存: ./data_cache/all_market_20200316_20260316.parquet，正在使用 PyArrow 飞速加载全市场数据...
正在构建量化计算引擎数据矩阵 (进行矩阵透视及停牌清洗)...
正在拉取基准指数 000300.SH 数据...

[System] 初始化智能体依赖库...

[System] 数据加载完毕，开始在【全市场环境】执行 Ralph Loop 智能迭代因子挖掘...

--- 启动第 1 轮迭代 ---
Agent 生成公式: Neg(Mul(SMA(Mul(TsRank(SignedPower(Div(Log(Add(EMA(Amount, 3), 0.0001)), TsRank(EMA(Amount,3), 20)), 0.5), 20), Sub(Sub(1, Corr(EMA(Amount, 5), Close, 10)), SignedPower(Div(Delta(Close, 1), Add(Abs(Sub(High, Delay(Close, 19))), 0.0001)), 0.5))), 5), Scale(CsRank(Close))))
评测结果: 是否成功=False, 指标={'Error': '选定时间段内的有效 IC 样本量不足'}

--- 启动第 2 轮迭代 ---
Agent 生成公式: Neg(TsRank(Corr(Div(Sub(Close, Open), Add(Sub(High, Low), 0.0001)), Log(Add(Abs(Delta(Volume, 1)), 1)), 10), 20))
评测结果: 是否成功=False, 指标={'Period': '20200316 to 20250316', 'IC': -0.0033, 'IR': -0.0682, 'IC_WinRate': 0.5398, 't_value': np.float64(-2.36), 'EXCESS_Return': 0.021, 'LS_Return': 0.0532, 'LS_Sharpe': np.float64(0.2785), 'Max_Drawdown': -0.1078, 'Monotonicity':

In [ ]:
"""
可以优化的方向：
1.用更好的大语言模型
2.采用更多的大模型，分属不同职位
3.多模态输入
4.增添算子和机器学习算子
5.增添因子评价
7.输入数据细粒度




"""

In [21]:
if __name__ == '__main__':
    # 1. 实例化引擎对象 (只用实例化一次，配置好基础路径)
    engine = FactorBacktestEngine(
        market_data_path="data_cache/all_market_20200316_20260316.parquet",
        local_factor_dir="all_market_factor_data"
    )
    TUSHARE_TOKEN = "1067ed1160b1dedeca68f122160aca415393dfd50f9f49ccbd1d0580"
    data_loader = TushareDataLoader(
        token=TUSHARE_TOKEN, 
        start_date="20200316", 
        end_date="20260316"
    )
    # 2. 执行基于公式的回测
    test_formula = "Neg(SMA(Mul(Div(Add(Sub(High, Low), Sub(Abs(Delta(Close, 1)), 0.0001)), Close), Power(Abs(SignedPower(Log(Div(Volume, Mean(Volume, 20))), 2)), 0.5)), 5))"

    returns, fees =engine.run_backtest(
        start_date="2020-03-16",
        end_date="2025-03-16",
        data_loader=data_loader,
        bench_code="000300.SH",     # <--- 传入 loader
        pool_code='000300.SH',       # <--- 指定回测时实时过滤沪深300
        factor_mode='formula',
        formula=test_formula,
        top_k=10,
        rebalance_days=5,
        max_stocks_in_bt=400
    )



🚀 初始化回测引擎 | 模式: FORMULA | 2020-03-16 -> 2025-03-16
📥 1. 加载本地行情库: data_cache/all_market_20200316_20260316.parquet
正在拉取基准指数 000300.SH 数据...
🌐 正在构建 000300.SH 历史动态成分股掩码矩阵...


/tmp/ipykernel_477030/2508148554.py:202: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_wide = mask_wide.reindex(index=target_index).ffill().fillna(False)


✅ 动态成分股矩阵构建完成！
⚙️ 2. 准备宽表并调用 GPU 计算引擎...
🔄 3. 正在将因子矩阵对齐并合并回长表...
🛡️ 4. 提取流动性最好的前 400 只股票以防内存溢出...
🚀 5. 启动 Backtrader (TopK=10, 调仓期=5天)...
💰 初始资金: 1000000.00
💰 最终资金: 890767.39

📊 交易费率与换手统计报告
💰 累计总手续费: 475,222.56 元
💸 累计总交易额: 316,815,042.24 元
🔄 日均换手率:   27.88 %
🌪️ 年化换手率:   70.26 倍
📉 手续费磨损率: 47.52% (占本金)
📊 6. 正在后台生成 QuantStats 评估报告...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

✅ 回测全流程完成！
请在浏览器中打开查看报告: /root/quant/Report_AI_Formula_Alpha_2020-03-16_to_2025-03-16.html


In [17]:

TUSHARE_TOKEN = "1067ed1160b1dedeca68f122160aca415393dfd50f9f49ccbd1d0580"
TARGET_DATE = '2024-12-09'
POOL_CODE = '000300.SH'  # 🎯 设定你想要的测试池，例如沪深300
TEST_FORMULA = "Neg(SMA(Mul(Div(Add(Sub(High, Low), Sub(Abs(Delta(Close, 1)), 0.0001)), Close), Power(Abs(SignedPower(Log(Div(Volume, Mean(Volume, 20))), 2)), 0.5)), 5))"
data_loader = TushareDataLoader(
        token=TUSHARE_TOKEN, 
        start_date="20200316", 
        end_date="20260316"
    )
# =========================================================
# 1. 基础数据准备 (假设 data_loader 已实例化)
# =========================================================
unpost_stock_data, stock_data = data_loader.fetch_daily_data()
benchmark_returns = data_loader.fetch_benchmark_returns(ts_code='000300.SH')
forward_returns = data_loader.build_forward_returns(stock_data['Close'], periods=1)

# =========================================================
# 🎯 2. 获取并构建动态股票池掩码
# =========================================================
print(f"🌐 正在获取 {POOL_CODE} 动态成分股掩码...")
pool_mask = data_loader.fetch_dynamic_pool_mask(
    pool_code=POOL_CODE, 
    target_index=forward_returns.index, 
    target_columns=forward_returns.columns
)

info_memory = InformationBase("all_market_info_base.json")
factor_assets = FactorBase("all_market_factor_base.json", "./all_market_factor_data/")

# =========================================================
# 3. 初始化引擎并注入 Mask
# =========================================================
print(f"⏳ 正在本地 GPU 引擎计算 {TEST_FORMULA} ...")
engine = EvaluationEngine(
    stock_data=stock_data, 
    forward_returns=forward_returns, 
    factor_base=factor_assets,
    benchmark_returns=benchmark_returns,
    pool_mask=pool_mask  # <--- 💡 核心：把掩码注入引擎
)

# 4. 调用引擎计算全量张量
factor_tensor = engine._calculate_factor(TEST_FORMULA)

# =========================================================
# 🎯 5. 强制应用掩码过滤 (非池内股票直接置为 NaN)
# =========================================================
if engine.pool_mask_tensor is not None:
    factor_tensor = torch.where(
        engine.pool_mask_tensor, 
        factor_tensor, 
        torch.tensor(float('nan'), device=engine.pool_mask_tensor.device)
    )

# 6. 转化为 DataFrame
factor_df = pd.DataFrame(
    factor_tensor.cpu().numpy() if hasattr(factor_tensor, 'cpu') else factor_tensor, 
    index=engine.index_dates, 
    columns=engine.columns_stocks
)

# 7. 提取特定日期的截面数据
if pd.to_datetime(TARGET_DATE) not in factor_df.index:
    raise ValueError(f"日期 {TARGET_DATE} 不在本地数据索引中，请检查是否为交易日！")

day_section = factor_df.loc[pd.to_datetime(TARGET_DATE)]

# 8. 清洗 NaN，从大到小排序，取前 50 名 (你会发现上榜的全部是沪深300成分股)
top_50_local = day_section.dropna().sort_values(ascending=False).head(20)

print(f"\n======== 本地引擎 {TARGET_DATE} 因子 Top 50 ({POOL_CODE}) ========")
for stock, score in top_50_local.items():
    print(f"{stock:^12} | {score:>10.6f}")
print("===================================================================")

命中本地缓存: ./data_cache/all_market_20200316_20260316.parquet，正在使用 PyArrow 飞速加载全市场数据...
正在构建量化计算引擎数据矩阵 (进行矩阵透视及停牌清洗)...
正在拉取基准指数 000300.SH 数据...
🌐 正在获取 000300.SH 动态成分股掩码...
🌐 正在构建 000300.SH 历史动态成分股掩码矩阵...
✅ 动态成分股矩阵构建完成！
⏳ 正在本地 GPU 引擎计算 Neg(SMA(Mul(Div(Add(Sub(High, Low), Sub(Abs(Delta(Close, 1)), 0.0001)), Close), Power(Abs(SignedPower(Log(Div(Volume, Mean(Volume, 20))), 2)), 0.5)), 5)) ...

======== 本地引擎 2024-12-09 因子 Top 50 (000300.SH) ========
 600089.SH   |  -0.001111
 601398.SH   |  -0.001442
 600023.SH   |  -0.001898
 000983.SZ   |  -0.002331
 601169.SH   |  -0.002457
 601872.SH   |  -0.002634
 600585.SH   |  -0.002909
 601166.SH   |  -0.003150
 000876.SZ   |  -0.003191
 002001.SZ   |  -0.003293
 600188.SH   |  -0.003382
 601988.SH   |  -0.003624
 600436.SH   |  -0.003726
 600905.SH   |  -0.003755
 600019.SH   |  -0.003760
 601766.SH   |  -0.003933
 002252.SZ   |  -0.003944
 601006.SH   |  -0.004068
 601600.SH   |  -0.004077
 601328.SH   |  -0.004193


/tmp/ipykernel_477030/2508148554.py:202: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_wide = mask_wide.reindex(index=target_index).ffill().fillna(False)


In [43]:
from jqdatasdk import *
jqdatasdk.auth("13881201408", "Du20041006@")
# 1. 设定测试日期和参数
TARGET_DATE = '2024-12-09'
WINDOW = 10
DATA_COUNT = 13 
# 2. 获取聚宽这一天全市场的股票列表
stocks = list(get_all_securities(date=TARGET_DATE).index)

print(f"⏳ 正在拉取 {TARGET_DATE} 之前 {WINDOW} 个交易日的【后复权】收盘价...")

# 3. 强制使用 fq='post' (后复权) 拉取刚好 10 天的数据
df = get_price(stocks, end_date=TARGET_DATE, count=DATA_COUNT, fields=['close', 'volume'], fq='post', panel=False)

# 数据透视
pivot_close = df.pivot(index='time', columns='code', values='close')
pivot_volume = df.pivot(index='time', columns='code', values='volume')

# 1. 计算 Delta(..., 3) 
# diff(3) 意味着今天减去 3天前
delta_close = pivot_close.diff(3)
delta_vol = pivot_volume.diff(3)

# 2. 计算 TsRank(..., 10)
# 取过去 10 天的数据进行滚动百分位排序 (0~1 之间)
tsrank_close = delta_close.rolling(10).rank(pct=True)
tsrank_vol = delta_vol.rolling(10).rank(pct=True)

# 3. 计算 Mul (两者相乘)，并只取最后一天(TARGET_DATE)的值
factor_res = tsrank_close.iloc[-1] * tsrank_vol.iloc[-1]

# 4. 排序提取 Top 10
top_10_jq = factor_res.dropna().sort_values(ascending=False).head(10)

print(f"\n======== 聚宽平台 {TARGET_DATE} 进阶因子 Top 10 ========")
for stock, score in top_10_jq.items():
    ts_format_stock = stock.replace('.XSHE', '.SZ').replace('.XSHG', '.SH')
    print(f"{ts_format_stock:^12} | {score:>10.6f}")
print("================================================================")

⏳ 正在拉取 2024-12-09 之前 10 个交易日的【后复权】收盘价...

======== 聚宽平台 2024-12-09 进阶因子 Top 10 ========
 000520.SZ   |   1.000000
 688665.SH   |   1.000000
 688685.SH   |   1.000000
 301296.SZ   |   1.000000
 688786.SH   |   1.000000
 688698.SH   |   1.000000
 002589.SZ   |   1.000000
 002590.SZ   |   1.000000
 301210.SZ   |   1.000000
 603285.SH   |   1.000000


In [8]:
pd.read_parquet("data_cache/all_market_20200316_20260316.parquet")

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor
7158927,000001.SZ,2020-03-16,14.45,14.46,13.75,13.75,14.52,-0.77,-5.3030,1406202.18,1975824.191,109.1694
7156619,000002.SZ,2020-03-16,29.46,29.46,28.29,28.54,29.45,-0.91,-3.0900,646614.59,1868241.398,148.4119
7156620,000004.SZ,2020-03-16,37.98,37.99,34.66,35.50,37.99,-2.49,-6.5544,70455.88,256794.896,4.0639
7158036,000005.SZ,2020-03-16,2.97,2.99,2.85,2.87,2.94,-0.07,-2.3810,104587.00,30500.385,9.2676
7159354,000006.SZ,2020-03-16,4.85,5.03,4.71,4.75,4.78,-0.03,-0.6276,251464.27,122778.381,35.4260
...,...,...,...,...,...,...,...,...,...,...,...,...
5476,920978.BJ,2026-03-13,26.68,27.05,26.41,26.68,26.75,-0.07,-0.2617,15501.57,41485.465,1.2885
5477,920981.BJ,2026-03-13,30.59,31.04,30.06,30.06,30.75,-0.69,-2.2439,5997.37,18316.393,1.4343
5478,920982.BJ,2026-03-13,186.58,190.00,185.11,186.31,187.30,-0.99,-0.5286,3774.28,70678.657,4.2831
5479,920985.BJ,2026-03-13,8.11,8.46,8.07,8.27,8.13,0.14,1.7220,84825.29,70421.869,1.6280
